# 19 · Verifica: i messaggi sono arrivati?
*Blocco 5 · Dati in Fabric. Passo di verifica (solo letture).*

Tre posti dove guardare, nell'ordine del percorso: l'**Hub** (li ha accettati e inoltrati?), la **Eventstream** (sorgente, operatori, destinazione: è tutto attivo e i campi sono quelli giusti?), l'**Eventhouse** (c'è la tabella, quali colonne ha?).

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

## Sull'Hub
I contatori di Azure Monitor arrivano con qualche minuto di ritardo: se sono a zero, riesegui la cella dopo un po'.
*Consegne alle route* = messaggi accettati moltiplicati per le route che li ricevono.

In [ ]:
if (-not $cfg -or -not $out) { throw "Mancano le variabili dello scenario: esegui prima la cella di caricamento (la prima cella di codice) di questo notebook." }
$cfg.HubNames | ForEach-Object { Get-HubMessageCounts -HubName $_ -ResourceGroup $cfg.Rg -Hours 1 } | Format-Table -AutoSize | Out-String

## Le route e gli endpoint

In [ ]:
foreach ($h in $cfg.HubNames) {
    "--- $h"
    az iot hub message-route list --hub-name $h --resource-group $cfg.Rg --query '[].{route:name, endpoint:endpointNames[0], attiva:isEnabled}' -o table
}

## La Eventstream
La topologia, letta con l'utente di `az login`: sorgenti, **operatori** (per esempio *ManageFields*, che può scartare i campi) e destinazioni, con il loro stato. Una destinazione *Paused* non scrive niente.

In [ ]:
$topo = az rest --resource https://api.fabric.microsoft.com --method get --url "https://api.fabric.microsoft.com/v1/workspaces/$($fabric.WorkspaceId)/eventstreams/$($fabric.EventstreamId)/topology" -o json --only-show-errors | ConvertFrom-Json
$topo.sources | ForEach-Object { "sorgente     : {0} ({1}) stato {2}" -f $_.name, $_.type, $_.status }
$topo.operators | ForEach-Object {
    $op = $_
    "operatore    : {0} ({1})" -f $op.name, $op.type
    $op.inputSchemas | ForEach-Object { "   schema in ingresso: " + (($_.schema.columns | ForEach-Object { $_.name }) -join ', ') }
    $op.properties.columns | ForEach-Object { "   colonna tenuta   : {0} ({1})" -f $_.alias, $_.type }
}
$topo.destinations | ForEach-Object { "destinazione : {0} ({1}) stato {2}, tabella {3}.{4}" -f $_.name, $_.type, $_.status, $_.properties.databaseName, $_.properties.tableName }

## L'Eventhouse
Le tabelle del database KQL e le colonne di `Events`, lette con una query (solo letture). Se la tabella non c'è, la cella lo dice. Se ci sono righe, mostra le colonne e le ultime cinque: sono i campi che arrivano davvero.

In [ ]:
$dest = $topo.destinations | Where-Object { $_.type -eq 'Eventhouse' } | Select-Object -First 1
$dbId = $dest.properties.itemId
$kdb = az rest --resource https://api.fabric.microsoft.com --method get --url "https://api.fabric.microsoft.com/v1/workspaces/$($fabric.WorkspaceId)/kqlDatabases/$dbId" -o json --only-show-errors | ConvertFrom-Json
$q = $kdb.properties.queryServiceUri
$tok = az account get-access-token --resource $q --query accessToken -o tsv --only-show-errors
function Invoke-Kusto($endpoint, $csl) {          # il database ha per nome l'id dell'item
    (Invoke-RestMethod -Method Post -Uri "$q/v1/rest/$endpoint" -Headers @{ Authorization = "Bearer $tok" } -ContentType 'application/json; charset=utf-8' `
        -Body (@{ db = $dbId; csl = $csl } | ConvertTo-Json -Compress)).Tables[0]
}
$tabelle = @((Invoke-Kusto 'mgmt' '.show tables').Rows | ForEach-Object { $_[0] })
if ($tabelle -notcontains $dest.properties.tableName) {
    "nel database '$($dest.properties.databaseName)' non c'e' la tabella '$($dest.properties.tableName)' (tabelle: $($tabelle -join ', ')): dalla Eventstream non e' arrivato niente."
} else {
    $t = $dest.properties.tableName
    "righe in ${t}: " + (Invoke-Kusto 'query' "$t | count").Rows[0][0]
    (Invoke-Kusto 'mgmt' ".show table $t cslschema").Rows | ForEach-Object { "colonne: " + $_[1] }
    $r = Invoke-Kusto 'query' "$t | take 5"
    $r.Rows | ForEach-Object { $row = $_; ($r.Columns | ForEach-Object { $i = [array]::IndexOf($r.Columns, $_); "{0}={1}" -f $_.ColumnName, $row[$i] }) -join ' | ' }
}

## I campi
L'Hub documenta uno schema **CloudEvents** per questa destinazione, ma l'anteprima dati della Eventstream mostra i campi del **payload** del device: `ts`, `deviceId`, `temperature`, `setpoint`, `payloadVersion`, `alarmState`. La tabella `Events` ha quelle colonne, con gli stessi nomi: una colonna con un nome diverso resta vuota. Se un operatore *ManageFields* tiene solo certi campi, arriva solo quello: si vede nella cella dell'operatore e nelle colonne dell'Eventhouse.

## Cosa abbiamo visto
- L'Hub scrive sulla Eventstream con la **sua identità**: nessuna chiave, ma l'identità di ogni Hub sta nel workspace Fabric.
- I messaggi che l'Hub accetta vengono inoltrati dalla route: i contatori lo mostrano.
- Quello che arriva in Fabric si guarda in Fabric: la sorgente, gli operatori, la destinazione, poi la tabella dell'Eventhouse.

Prossimo notebook: **99**, fine sessione: elimina tutto (solo alla fine della demo).